<span style="color:red;font-size:2em;font-weight:bold"> PARTIE 1 - Analyse exploratoire (part2) - Nettoyage et aggregation séquentiel</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modules </span>

In [ ]:
# Roots
import gc
import numpy as np
import pandas as pd
# import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Stats
from scipy.stats import zscore, chi2_contingency, f_oneway, chi2

In [2]:
# Sert à éviter les Warnings avec les transformations sur des vues en transformant 
# ces warning en erreur obligeant ainsi à ne travailler que sur des copies ou les originaux.

pd.set_option('mode.chained_assignment','raise')

In [3]:
# Module pour recharger un module sans redemarrer le kernel
# import importlib
%load_ext autoreload
%autoreload 2

In [4]:
# Ajoute le dossier datas_manipulation au sys.path. Remarque ne pas oublier le __init__.py dans le dossier datas_manipulation
import sys
# root_path = Path(__file__).resolve().parents[1] # Ne fonctionne pas sur notebook
root_path = Path.cwd().parent
sys.path.append(str(root_path))


In [ ]:
# Fonctions personnelles
from notebooks.datas_manipulation.quick_clean_datas import (
    drop_empty_columns, clean_infinites, drop_col_with_unique_value
)
from notebooks.datas_manipulation.datas_assembler import merging_data
from notebooks.datas_manipulation.memory_optimizer import optimize_dtypes, float_to_int
from notebooks.datas_manipulation.export_datas import export_datas

from notebooks.utils.feature_aggregator import agg_features, agg_columns

In [ ]:
# Paramètres globaux

# Création dossier results
save_path = root_path.joinpath('datas/results')
Path.mkdir(save_path,exist_ok = True)

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Datasets </span>

In [7]:
# Chemin du dossier de données brut
datas_path = root_path /'datas'/'raw_datas'/'Projet+Mise+en+prod+-+home-credit-default-risk'

<span style="color:blue;font-weight:bold"> Rappel des features </span>

In [8]:
# fichier de description
df_description = pd.read_csv(datas_path/"HomeCredit_columns_description.csv",encoding='latin1')
(df_description[["Row","Description"]].sort_values(by='Row')).T

,176,138,9,177,153,178,8,154,130,132,...,88,89,32,181,46,74,60,47,75,61
Row,AMT_ANNUITY,AMT_ANNUITY,AMT_ANNUITY,AMT_APPLICATION,AMT_BALANCE,AMT_CREDIT,AMT_CREDIT,AMT_CREDIT_LIMIT_ACTUAL,AMT_CREDIT_MAX_OVERDUE,AMT_CREDIT_SUM,...,TOTALAREA_MODE,WALLSMATERIAL_MODE,WEEKDAY_APPR_PROCESS_START,WEEKDAY_APPR_PROCESS_START,YEARS_BEGINEXPLUATATION_AVG,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_AVG,YEARS_BUILD_MEDI,YEARS_BUILD_MODE
Description,Annuity of previous application,Annuity of the Credit Bureau credit,Loan annuity,For how much credit did client ask on the prev...,Balance during the month of previous credit,Final credit amount on the previous applicatio...,Credit amount of the loan,Credit card limit during the month of the prev...,Maximal amount overdue on the Credit Bureau cr...,Current credit amount for the Credit Bureau cr...,...,Normalized information about building where th...,Normalized information about building where th...,On which day of the week did the client apply ...,On which day of the week did the client apply ...,Normalized information about building where th...,Normalized information about building where th...,Normalized information about building where th...,Normalized information about building where th...,Normalized information about building where th...,Normalized information about building where th...


In [9]:
print(f"nombre de features uniques:{df_description["Row"].nunique()}")
# on regroupe suivant Row est on créée une liste des tables associées. On obtient
# une Series avec les Row en index et la liste des tables en valeurs
duplicated_features = df_description.groupby('Row')['Table'].apply(list).reset_index()
# on ne garde que les features dupliquées
duplicated_features = duplicated_features[duplicated_features['Table'].map(len) > 1]

final_view = duplicated_features.set_index('Row').T
display(final_view)

nombre de features uniques:196


Row,AMT_ANNUITY,AMT_CREDIT,AMT_GOODS_PRICE,HOUR_APPR_PROCESS_START,MONTHS_BALANCE,NAME_CONTRACT_STATUS,NAME_CONTRACT_TYPE,NAME_TYPE_SUITE,SK_BUREAU_ID,SK_DPD,SK_DPD_DEF,SK_ID_CURR,SK_ID_PREV,WEEKDAY_APPR_PROCESS_START
Table,"[application_{train|test}.csv, bureau.csv, pre...","[application_{train|test}.csv, previous_applic...","[application_{train|test}.csv, previous_applic...","[application_{train|test}.csv, previous_applic...","[bureau_balance.csv, POS_CASH_balance.csv, cre...","[POS_CASH_balance.csv, credit_card_balance.csv...","[application_{train|test}.csv, previous_applic...","[application_{train|test}.csv, previous_applic...","[bureau.csv, bureau_balance.csv]","[POS_CASH_balance.csv, credit_card_balance.csv]","[POS_CASH_balance.csv, credit_card_balance.csv]","[application_{train|test}.csv, bureau.csv, POS...","[POS_CASH_balance.csv, credit_card_balance.csv...","[application_{train|test}.csv, previous_applic..."


<span style="color:red">Remarque: erreur dans le nommage SK_BUREAU_ID dans le fichier de description ==> dans la data c'est bien SK_ID_BUREA</span>

On retrouve une description des fichiers ainsi qu'un diagramme de relation sur https://www.kaggle.com/c/home-credit-default-risk/data.


**Rappel des fichiers**

En comptant le fichier descriptif, on a 10 fichiers (c'est plus que le diagramme car train|test ensemble tandis que le fichier *HomeCredit_columns_description* et *sample_submission* non plus car l'un décrit les features et l'autre). Les fichiers sont reliés par le SK_ID_CURR (l'ID client CHEZ HOME CREDIT) et une liaison complémentaire se fait avec les autres ID (SK_ID_BUREAU pour bureau et bureau_balance / SK_ID_PREV pour previous_application avec POS_CASH_balance, installments_payments et credit_card_balance). Pour ce qui est de leur contenu:
- **application_train/test**: Les données principales (Démographie, revenus, montant du prêt). Une ligne = Un prêt.
- **bureau**: Données de tous les emprunts enregistrés au Bureau du Crédit (toutes les institutions incluant aussi Home Credit).
- **bureau_balance**: Historique mensuel des crédits (état de remboursement).
- **previous_application**: Toutes les demandes de prêts faites par le client chez Home Credit par le passé.
- **POS_CASH_balance**: Historique mensuel des soldes des anciens prêts (Point of Sale et Cash).
- **installments_payments**: Historique de chaque paiement réalisé par rapport aux anciens prêts (réussi ou echoué).
- **credit_card_balance**: Historique mensuel de l'utilisation des cartes de crédit du client.


<span style="color:black;font-size:1em;background-color:yellow"> CREDIT BUREAU EST UNE INSTITUTION QUI RECENSE LES PRETS CONTRAIREMENT A HOME CREDIT QUI EST LA BANQUE PRETEUSE ICI!!!</span>

<span style="color:blue;font-weight:bold"> Dataframe prinipale: df_train_test</span>

In [10]:
miss = 0.8
miss_percent = int(miss*100)

In [11]:
df_train_test = pd.read_parquet(datas_path/f"train_test{miss_percent}_cleaned.parquet")

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de la branche bureau</span>

Comme il a été dit précédemment au vu du nombre de lignes qui dépasse très largement le nombre d'observations train/test, en partant du postulat que ce dépassement (de plusieurs millions tout de même pour certains fichiers) est majoritairement lié au fait que plusieurs lignes concernent un même client (avec éventuellement un peu de doublons), on va devoir regarder séparemment leur contenu et aggreger afin de n'avoir qu'une observation par client.

- **bureau** contient l'ensemble des prêts déclarés dans une institution des clients. Chaque ligne correspond a un crédit et caractérisée par deux ID:
    - SK_ID_CURR: l'ID du client chez Home Credit
    - SK_ID_BUREAU: L'ID d'un prêt enregistré chez Credit Bureau
- **bureau_balance** contient l'historique mensuel de tous les prêts chez Credit Bureau (CrB). Chaque ligne correspond à l'état mensuel d'un prêt au sein de CrB caractérisée par:
    - SK_ID_BUREAU
    - MONTHS_BALANCE: $\leq 0$ avec 0, le mois courant
    - STATUS: Etat du remboursement à ce jour [0-5,C,X], 0 a 5 pour un retard sur le remboursemnt (plus la valeur est elevée plus il y a du retard avec 0 pas de retard et 5 pour prêt revendu), C pour un prêt cloturé et X pour dire statut inconnu. 



In [ ]:
steps = ["bureau", "prev_app", "inst_pay", "installements", "credit_card"]
ID_list = ["SK_ID_CURR", "SK_ID_PREV", "SK_ID_BUREAU"]

In [ ]:
temp_file_path_bureau = datas_path/steps[0]/f"df_train_test_{steps[0]}.parquet"

if temp_file_path_bureau.exists():
    df_train_test = pd.read_parquet(temp_file_path_bureau)
else:
    # On importe le premier fichier sur lequel on aggrege la donnée d'abord puis 
    # qui sera fusionné dans le second (et garbage collecté via la fonction d'agg).

    # Bureau Balance
    df_bureau_balance = pd.read_parquet(datas_path/f'bureau_balance{miss_percent}_cleaned.parquet')
    # Agrégation de Bureau Balance
    df_bureau_balance_agg = agg_features(
        df_bureau_balance, 'SK_ID_BUREAU', 'BB', drop_columns=ID_list
    )


    # Bureau (On joint BB dedans d'abord)
    df_bureau = pd.read_parquet(datas_path / f"bureau{miss_percent}_cleaned.parquet")
    
    # ============== FEATURE ENGINEERING avant agg ==================
    # Capacité d'emprunt residuelle =1 ==> sa capacité d'emprunt est entierement conso
    df_bureau["RESIDUAL_CREDIT_CAPACITY"] = \
        df_bureau['AMT_CREDIT_SUM_DEBT']/df_bureau['AMT_CREDIT_SUM_LIMIT']
    
    # ========================================================
    
    # On attache les infos de balance à bureau puis on le garbage collect
    df_bureau = merging_data(df_bureau, df_bureau_balance_agg, on='SK_ID_BUREAU', how='left')


    # Agrégation finale pour n'avoir qu'une ligne par client
    df_bureau_agg = agg_features(df_bureau, 'SK_ID_CURR', 'BUREAU', drop_columns=ID_list)


    # Fusion avec le df_test_train
    df_train_test = merging_data(df_train_test, df_bureau_agg, on='SK_ID_CURR', how='left')
    
    # on réoptimise le type des colonnes
    df_train_test = optimize_dtypes(df_train_test)
    # on sauvegarde temporairement le df_bureau nettoyé
    export_datas(df_train_test, datas_path, step = steps[0], prefix = "df_train_test_")

In [ ]:
# df_train_test.head(10)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_CREDIT_TYPE_Interbank credit,BUREAU_CREDIT_TYPE_Loan for business development,BUREAU_CREDIT_TYPE_Loan for purchase of shares (margin lending),BUREAU_CREDIT_TYPE_Loan for the purchase of equipment,BUREAU_CREDIT_TYPE_Loan for working capital replenishment,BUREAU_CREDIT_TYPE_Microloan,BUREAU_CREDIT_TYPE_Mobile operator loan,BUREAU_CREDIT_TYPE_Mortgage,BUREAU_CREDIT_TYPE_Real estate loan,BUREAU_CREDIT_TYPE_Unknown type of loan
0,100002.0,1.0,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,100003.0,0.0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004.0,0.0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006.0,0.0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007.0,0.0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,100008.0,0.0,Cash loans,M,N,Y,0,99000.0,490495.5,27517.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,100009.0,0.0,Cash loans,F,Y,Y,1,171000.0,1560726.0,41301.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,100010.0,0.0,Cash loans,M,Y,Y,0,360000.0,1530000.0,42075.0,...,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,100011.0,0.0,Cash loans,F,N,Y,0,112500.0,1019610.0,33826.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,100012.0,0.0,Revolving loans,M,N,Y,0,135000.0,405000.0,20250.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# df_train_test.info()

On peut voir qu'on a pratiquement doubler le nombre de features (mais on s'est évité les 27 millions de ligne de bureau_balance). On avait pu prendre connaissance précédemment des features doublons (donc il faut attendre la fusion pour manipuler) ainsi que la description les concernant.
- on peut donc réaliser des features engineering
- on peut aggréger des colonnes ensemble
- supprimer des colonnes vides (80 ou 90%?)

In [16]:
# on peut suppr l'ID BUREAU
df_train_test.drop(columns='SK_ID_BUREAU',inplace=True,errors='ignore')

<span style="color:blue;font-weight:bold"> Aggregation des FLAG </span>

In [17]:
# Possession (car & realty) ==> un proprietaire réduit le risque de défaut in fine
df_train_test = agg_columns(df_train_test, 'POSSESSION', ["FLAG_OWN_CAR","FLAG_OWN_REALTY"])
# Conversion des NN, YN, NY, YY en 0,1,2 et 3 ==> [car,real estate]
df_train_test['POSSESSION'] = (
    (df_train_test['POSSESSION']== 'YN').astype('int8')  * 1
    + (df_train_test['POSSESSION'] == 'NY').astype('int8')  * 2
    + (df_train_test['POSSESSION'] == 'YY').astype('int8')  * 3
)

In [18]:
# Documentation (2 a 21) ==> Nombre de doc admin rempli par le client, moins de risque si plus
flag_doc_list = [col for col in df_train_test.columns if col.startswith('FLAG_DOCUMENT')]
df_train_test=agg_columns(df_train_test,'FILLED_DOC_COUNT',flag_doc_list)

In [19]:
# EXT_SOURCE (1,2 et 3)
df_train_test=agg_columns(
    df_train_test,'EXT_SOURCE_COUNT',['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'])

In [20]:
# Moyn de communication fourni (mail, tel, mobile..)
contact_ways_list=[
    'FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE','FLAG_CONT_MOBILE',
    'FLAG_PHONE','FLAG_EMAIL'
]
df_train_test=agg_columns(df_train_test,'CONTACT_WAYS_COUNT',contact_ways_list)

In [21]:
# Informations erronees qui rend le client suspect
mismatch_info_list=[
    'REG_REGION_NOT_LIVE_REGION',
    'REG_CITY_NOT_LIVE_CITY',
    'REG_CITY_NOT_WORK_CITY'
]
df_train_test=agg_columns(df_train_test,'MISMATCH_INFO_COUNT', mismatch_info_list)

<span style="color:blue;font-weight:bold"> Feature engineering  </span>

In [22]:
# Taux d'endettement
df_train_test['DEBT_RATIO'] = \
    df_train_test['AMT_ANNUITY'] / df_train_test['AMT_INCOME_TOTAL']

# Vitesse de remboursment
df_train_test['PAYMENT_RATE'] = \
    df_train_test['AMT_ANNUITY'] / df_train_test['AMT_CREDIT']

# Solvabilité
df_train_test['SOLVABILITY'] = \
    df_train_test['AMT_INCOME_TOTAL'] / df_train_test['AMT_CREDIT']

# Temps de travail effectif relative
df_train_test['DAYS_EMPLOYED_RATIO'] = \
    df_train_test['DAYS_EMPLOYED'] / df_train_test['DAYS_BIRTH']

In [23]:
# impossible pour le moment de faire des chi2, anova , heatmap etc... donc corr simple
# on va suppr les features très proche de zero
correlations = abs(df_train_test.select_dtypes(include=[np.number]).corr()['TARGET']).sort_values()

In [24]:
print(correlations)

BUREAU_AMT_CREDIT_SUM_OVERDUE_MIN       0.000003
BUREAU_RESIDUAL_CREDIT_CAPACITY_MEAN    0.000007
BUREAU_RESIDUAL_CREDIT_CAPACITY_MIN     0.000039
BUREAU_CNT_CREDIT_PROLONG_MIN           0.000182
BUREAU_AMT_CREDIT_SUM_DEBT_MIN          0.000242
                                          ...   
BUREAU_CREDIT_ACTIVE_Closed             0.079369
BUREAU_BB_MONTHS_BALANCE_MIN_MEAN       0.089036
BUREAU_DAYS_CREDIT_MEAN                 0.089728
EXT_SOURCE_COUNT                        0.173322
TARGET                                  1.000000
Name: TARGET, Length: 178, dtype: float64


La correlation simple montre qu'aucune feature numériques ne dépasse les 0.2 mais beaucoup de features même parmi les plus faibles ont une importance métier non négligeable, on ne va pas procéder à une élimination pour le moment donc.

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de la branche previous_application</span>

A la différence de la branche bureau pour laquelle la sous-branche, bureau_balance n'est liée qu'à bureau (via SK_ID_BUREAU), les fichiers previous_application, POS_CASH_balance, installments_payments et credit_card_balance sont tous liés directement à train_test c'est pourquoi on peut se permettre de tous traiter et merger directement vers train_test. De plus, moins on introduit d'étapes intermédiaire modifiant la donnée moins il y aura de biais. Et pour finir le point central reste la gestion de mémoire, installments_payments et credit_card_balance sont les fichiers les plus lourds, ainsi aggregation et jointure seront particulièrement gourmandes et pour minimiser le coût, il sera préférable de procéder ainsi.

In [ ]:
temp_file_path_prev_app = datas_path/steps[1]/f"df_train_test_{steps[1]}.parquet"

if temp_file_path_prev_app.exists():
    del df_train_test
    gc.collect()
    df_train_test = pd.read_parquet(temp_file_path_prev_app)
else:
    prev_app = pd.read_parquet(datas_path/f"previous_application{miss_percent}_cleaned.parquet")
    # On peut supprimer SK_ID_PREV ici car inutile, on lie les fichiers sur train_test
    # + previous application est la source si on liait avec SK_ID_PREV, diff pour les autres
    prev_app.drop(columns='SK_ID_PREV', inplace=True, errors="ignore")
    
    # ======== FEATURE ENGINEERING a faire ici pour anticiper aggregations =========
    # pourcentage effectif de la précédente demande de prêt.
    # <1 signifie que le profil n'était pas assez bon.  
    prev_app['CREDIT_APP_RATIO'] = prev_app['AMT_CREDIT'] / prev_app['AMT_APPLICATION']
    
    # Retard sur la fin de contrat. <1 = retard = risque (<1 car valeurs negatives)
    prev_app['TERMINATION_DELAYED'] = prev_app['DAYS_TERMINATION'] - prev_app['DAYS_LAST_DUE']
    # ===========================================================================
    
    # aggregation de previous application
    prev_agg = agg_features(prev_app, 'SK_ID_CURR', 'PREV', drop_columns=ID_list)
    
    # Fusion avec le df_test_train
    df_train_test = merging_data(df_train_test, prev_agg, on='SK_ID_CURR', how='left')
    
    # on réoptimise le type des colonnes
    df_train_test = optimize_dtypes(df_train_test)
    # on sauvegarde temporairement le df_previous_application nettoyé
    export_datas(df_train_test, datas_path, step = steps[1], prefix = "df_train_test_")

In [ ]:
# df_train_test.head(10)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,PREV_PRODUCT_COMBINATION_Cash X-Sell: low,PREV_PRODUCT_COMBINATION_Cash X-Sell: middle,PREV_PRODUCT_COMBINATION_POS household with interest,PREV_PRODUCT_COMBINATION_POS household without interest,PREV_PRODUCT_COMBINATION_POS industry with interest,PREV_PRODUCT_COMBINATION_POS industry without interest,PREV_PRODUCT_COMBINATION_POS mobile with interest,PREV_PRODUCT_COMBINATION_POS mobile without interest,PREV_PRODUCT_COMBINATION_POS other with interest,PREV_PRODUCT_COMBINATION_POS others without interest
0,100002.0,1.0,Cash loans,M,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1.0,0.0
1,100003.0,0.0,Cash loans,F,0,270000.0,1293502.5,35698.5,1129500.0,Family,...,0.333333,0.0,0.333333,0.000000,0.333333,0.0,0.000000,0.0,0.0,0.0
2,100004.0,0.0,Revolving loans,M,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,1.0,0.0,0.0
3,100006.0,0.0,Cash loans,F,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,...,0.222222,0.0,0.111111,0.000000,0.111111,0.0,0.000000,0.0,0.0,0.0
4,100007.0,0.0,Cash loans,M,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,...,0.000000,0.5,0.166667,0.000000,0.000000,0.0,0.166667,0.0,0.0,0.0
5,100008.0,0.0,Cash loans,M,0,99000.0,490495.5,27517.5,454500.0,"Spouse, partner",...,0.000000,0.2,0.400000,0.000000,0.000000,0.0,0.200000,0.0,0.0,0.0
6,100009.0,0.0,Cash loans,F,1,171000.0,1560726.0,41301.0,1395000.0,Unaccompanied,...,0.000000,0.0,0.714286,0.142857,0.000000,0.0,0.142857,0.0,0.0,0.0
7,100010.0,0.0,Cash loans,M,0,360000.0,1530000.0,42075.0,1530000.0,Unaccompanied,...,0.000000,0.0,0.000000,0.000000,0.000000,1.0,0.000000,0.0,0.0,0.0
8,100011.0,0.0,Cash loans,F,0,112500.0,1019610.0,33826.5,913500.0,Children,...,0.000000,0.0,0.250000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
9,100012.0,0.0,Revolving loans,M,0,135000.0,405000.0,20250.0,405000.0,Unaccompanied,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.250000,0.0,0.0,0.0


In [ ]:
# df_train_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 348 entries, SK_ID_CURR to PREV_PRODUCT_COMBINATION_POS others without interest
dtypes: float32(94), float64(226), int16(2), int64(3), int8(8), object(15)
memory usage: 795.0+ MB


<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de POS_CASH_balance</span>

Ce fichier décrit l'historique mensuel des crédits POS (Crédit obtenu sur un lieu de vente ==> **credit à la consommation?**) et prêt en cash.
A l'instar de bureau_balance, il faut d'abord aggreger les observations via SK_ID_PREV pour avoir des lignes suivant UN CONTRAT de prêt puis rassembler suivant SK_ID_CURR pour avoir une ligne PAR CLIENT (ce sera pareil pour instalment et credit_card).

In [ ]:
temp_file_path_pos_cash = datas_path/steps[2]/f"df_train_test_{steps[2]}.parquet"

if temp_file_path_pos_cash.exists():
    del df_train_test
    gc.collect()
    df_train_test = pd.read_parquet(temp_file_path_pos_cash)
else:
    pos_cash = pd.read_parquet(datas_path/f"POS_CASH_balance{miss_percent}_cleaned.parquet")
    
    # ======== FEATURE ENGINEERING a faire ici pour anticiper aggregations =========
    # pourcentage de completion du remboursment. 1 = pret remboursé
    pos_cash['DEBT_COMPLETION_RATIO'] = \
        pos_cash['CNT_INSTALMENT_FUTURE'] / pos_cash['CNT_INSTALMENT']
    # ===========================================================================
    
    # agg interne suivant SK_ID_PREV
    pos_cash_agg = agg_features(pos_cash, 'SK_ID_PREV', 'PC_ID_PREV', ID_list)
    # on créée un dataframe sur les ID CURR et PREV afin de pouvoir raccorder
    # Client et contrat (SK_ID_CURR ayant été perdu lors du agg_features)
    pos_cash_ID = pos_cash[['SK_ID_CURR','SK_ID_PREV']].drop_duplicates()
    # On merge suivant SK_ID_PREV
    pos_cash_agg = merging_data(pos_cash_agg,pos_cash_ID,"SK_ID_PREV", 'left')
    # On supprime aussi pos_cash, ID ayant été supprimer dans merging data
    del pos_cash
    gc.collect()
    
    # agg interne suivant SK_ID_CURR
    pos_cash_agg.drop(columns='SK_ID_PREV', inplace=True, errors='ignore')
    pos_cash_agg = agg_features(pos_cash_agg, 'SK_ID_CURR', 'PC', ID_list)
    
    # Fusion avec le df_test_train
    df_train_test = merging_data(df_train_test, pos_cash_agg, on='SK_ID_CURR', how='left')
    
    # on réoptimise le type des colonnes
    df_train_test = optimize_dtypes(df_train_test)
    # on sauvegarde temporairement le df_previous_application nettoyé
    export_datas(df_train_test, datas_path, step = steps[2], prefix = "df_train_test_")

**Avec la double aggregation, on remarquera que les colonnes aggrégée auront des titre du type "MEAN_MAX", "MIN_MIN"... La première agg décrit par contrat (moyenne des contrats suivant ID_PREV) et la seconde agg identifie le client (MEAN_MAX pour le maximum suivant la moyenne des contrats). Par exemple un retard MEAN_MAX signifierait qu'on regarde le maximum suivant la moyenne des retard du client**

In [29]:
df_train_test.head(10)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,PC_NAME_CONTRACT_STATUS_Completed_MIN,PC_NAME_CONTRACT_STATUS_Demand_MEAN,PC_NAME_CONTRACT_STATUS_Demand_MAX,PC_NAME_CONTRACT_STATUS_Demand_MIN,PC_NAME_CONTRACT_STATUS_Returned to the store_MEAN,PC_NAME_CONTRACT_STATUS_Returned to the store_MAX,PC_NAME_CONTRACT_STATUS_Returned to the store_MIN,PC_NAME_CONTRACT_STATUS_Signed_MEAN,PC_NAME_CONTRACT_STATUS_Signed_MAX,PC_NAME_CONTRACT_STATUS_Signed_MIN
0,100002.0,1.0,Cash loans,M,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0.000000,0.0,0.0,0.0,0.000000,0.000,0.0,0.000000,0.000000,0.0
1,100003.0,0.0,Cash loans,F,0,270000.0,1293502.5,35698.5,1129500.0,Family,...,0.000000,0.0,0.0,0.0,0.000000,0.000,0.0,0.000000,0.000000,0.0
2,100004.0,0.0,Revolving loans,M,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,...,0.250000,0.0,0.0,0.0,0.000000,0.000,0.0,0.000000,0.000000,0.0
3,100006.0,0.0,Cash loans,F,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,...,0.000000,0.0,0.0,0.0,0.041667,0.125,0.0,0.000000,0.000000,0.0
4,100007.0,0.0,Cash loans,M,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,...,0.000000,0.0,0.0,0.0,0.000000,0.000,0.0,0.015385,0.076923,0.0
5,100008.0,0.0,Cash loans,M,0,99000.0,490495.5,27517.5,454500.0,"Spouse, partner",...,0.018182,0.0,0.0,0.0,0.000000,0.000,0.0,0.031250,0.125000,0.0
6,100009.0,0.0,Cash loans,F,1,171000.0,1560726.0,41301.0,1395000.0,Unaccompanied,...,0.000000,0.0,0.0,0.0,0.000000,0.000,0.0,0.000000,0.000000,0.0
7,100010.0,0.0,Cash loans,M,0,360000.0,1530000.0,42075.0,1530000.0,Unaccompanied,...,0.090909,0.0,0.0,0.0,0.000000,0.000,0.0,0.000000,0.000000,0.0
8,100011.0,0.0,Cash loans,F,0,112500.0,1019610.0,33826.5,913500.0,Children,...,0.022222,0.0,0.0,0.0,0.000000,0.000,0.0,0.000000,0.000000,0.0
9,100012.0,0.0,Revolving loans,M,0,135000.0,405000.0,20250.0,405000.0,Unaccompanied,...,0.043478,0.0,0.0,0.0,0.000000,0.000,0.0,0.000000,0.000000,0.0


In [30]:
# Modif des noms car trop long
# df_train_test.columns=df_train_test.columns.str.replace(r'^PC_ID_CURR_PC_ID_PREV_','PC_',regex=True)
# df_train_test.head()

In [ ]:
# df_train_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 444 entries, SK_ID_CURR to PC_NAME_CONTRACT_STATUS_Signed_MIN
dtypes: float32(121), float64(295), int16(2), int64(3), int8(8), object(15)
memory usage: 1019.3+ MB


In [ ]:
# df_train_test = optimize_dtypes(df_train_test)
# df_train_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 444 entries, SK_ID_CURR to PC_NAME_CONTRACT_STATUS_Signed_MIN
dtypes: float32(415), float64(1), int16(2), int8(11), object(15)
memory usage: 612.6+ MB


<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de instalments_payments</span>

On passe maintenant aux fichiers les plus lourds du jeu à commencer par instalments_payments.
Ce fichier recense l'ensemble des crédits souscrit chez Home Crédit, chaque ligne se référant à un paiement sur un prết validé ou non.

In [ ]:
temp_file_path_inst_pay = datas_path/steps[3]/f"df_train_test_{steps[3]}.parquet"

if temp_file_path_inst_pay.exists():
    del df_train_test
    gc.collect()
    df_train_test = pd.read_parquet(temp_file_path_inst_pay)
else:
    inst_pay = pd.read_parquet(datas_path/f"installments_payments{miss_percent}_cleaned.parquet")
    
    # ======== FEATURE ENGINEERING a faire ici pour anticiper aggregations =========
    # nombre de jours de retard suivant la date d'echeance
    inst_pay['LATE_PAYMENT'] = inst_pay['DAYS_ENTRY_PAYMENT'] - inst_pay['DAYS_INSTALMENT']
    # remboursment partiel(<1) ou total (1)
    inst_pay['PAY_INST_RATIO'] = inst_pay['AMT_PAYMENT']/inst_pay['AMT_INSTALMENT']
    # ===========================================================================
    
    # agg interne suivant SK_ID_PREV
    inst_pay_agg = agg_features(inst_pay, 'SK_ID_PREV', 'IP_ID_PREV', ID_list)
    # on créée un dataframe sur les ID CURR et PREV afin de pouvoir raccorder
    # Client et contrat (SK_ID_CURR ayant été perdu lors du agg_features)
    inst_pay_ID = inst_pay_agg[['SK_ID_CURR','SK_ID_PREV']].drop_duplicates()
    # On merge suivant SK_ID_PREV
    inst_pay_agg = merging_data(inst_pay_agg,inst_pay_ID,"SK_ID_PREV", 'left')
    # On supprime aussi pos_cash, ID ayant été supprimer dans merging data
    del inst_pay
    gc.collect()
    
    # agg interne suivant SK_ID_CURR
    inst_pay_agg.drop(columns='SK_ID_PREV', inplace=True, errors='ignore')
    inst_pay_agg = agg_features(inst_pay_agg, 'SK_ID_CURR', 'IP', ID_list)
    
    # Fusion avec le df_test_train
    df_train_test = merging_data(df_train_test, inst_pay_agg, on='SK_ID_CURR', how='left')
    
    # on réoptimise le type des colonnes
    df_train_test = optimize_dtypes(df_train_test)
    # on sauvegarde temporairement le df_previous_application nettoyé
    export_datas(df_train_test, datas_path, step = steps[3], prefix = "df_train_test_")

In [ ]:
# df_train_test.head(10)

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de credit_card_balance</span>

Ce fichier contient l'ensemble des transactions mensuels des comptes bancaires associées au cartes de crédit des clients, chaque ligne représentant l'état d'un compte.

<span style="color:red"> **Information pour moi-même**: La carte bancaire ici ne fonctionne pas de la même façon qu'en France. 
- En France, une carte de crédit/carte de débit (carte habituelle en France) est lié a un compte courant contenant une somme d'argent que le proprietaire va pouvoir dépenser (débit). Tant que la somme possédée est supérieur aux dépenses, on est en débit par contre si on dépasse (pas possible avec une carte de débit par contre), on risque des intérêts car on emprunte à la banque (cf agyo) est alors en découvert.
- Le fonctionnement aux USA est différent, la carte de crédit n'est pas lié au compte bancaire et c'est la banque qui prête une somme d'argent définie. Tant que cette somme n'est pas dépassée, on n'est pas considéré à découvert (pas d'intérêt?) par contre à la fin du mois, la banque demande a ce qu'un montant minmum de la dépense soit remboursée (AMT_INST_MIN_REGULAR), la dette restante étant transféré a mois suivant et dans l'hypothèse où même ce montant min ne peut être payer le client est alors en défaut de paiement etc...  </span>

In [ ]:
temp_file_path_credit_card = datas_path/steps[4]/f"df_train_test_{steps[4]}.parquet"

if temp_file_path_credit_card.exists():
    del df_train_test
    gc.collect()
    df_train_test = pd.read_parquet(temp_file_path_credit_card)
else:
    credit_card = pd.read_parquet(datas_path/f"credit_card_balance{miss_percent}_cleaned.parquet")
    
    # ======== FEATURE ENGINEERING a faire ici pour anticiper aggregations =========
    # Getion du plafond du client. =1, plafond atteint pour le mois donné, client a risque
    credit_card["BALANCE_RATIO"] = \
        credit_card['AMT_BALANCE']/credit_card['AMT_CREDIT_LIMIT_ACTUAL']
    # Comment le client gère sa dette de credit. >=1 roule/annule sa dette, <1 n'arrive plus
    credit_card['REVOLVING_RATIO'] = \
        credit_card['AMT_PAYMENT_TOTAL_CURRENT'] / credit_card['AMT_INST_MIN_REGULAR']
    # Depenses. pareil que balance ratio mais pour le mois courant
    credit_card["RESIDUAL_USE_CAPACITY"] = \
        credit_card['AMT_DRAWINGS_CURRENT'] / credit_card['AMT_CREDIT_LIMIT_ACTUAL']
    # ===========================================================================
    
    # agg interne suivant SK_ID_PREV
    credit_card_agg = agg_features(credit_card, 'SK_ID_PREV', 'CC_ID_PREV', ID_list)
    # on créée un dataframe sur les ID CURR et PREV afin de pouvoir raccorder
    # Client et contrat (SK_ID_CURR ayant été perdu lors du agg_features)
    credit_card_ID = credit_card_agg[['SK_ID_CURR','SK_ID_PREV']].drop_duplicates()
    # On merge suivant SK_ID_PREV
    credit_card_agg = merging_data(credit_card_agg,credit_card_ID,"SK_ID_PREV", 'left')
    # On supprime aussi pos_cash, ID ayant été supprimer dans merging data
    del credit_card
    gc.collect()
    
    # agg interne suivant SK_ID_CURR
    credit_card_agg.drop(columns='SK_ID_PREV', inplace=True, errors='ignore')
    credit_card_agg = agg_features(credit_card_agg, 'SK_ID_CURR', 'CC', ID_list)
    
    # Fusion avec le df_test_train
    df_train_test = merging_data(df_train_test, credit_card_agg, on='SK_ID_CURR', how='left')
    
    # on réoptimise le type des colonnes
    df_train_test = optimize_dtypes(df_train_test)
    # on sauvegarde temporairement le df_previous_application nettoyé
    export_datas(df_train_test, datas_path, step = steps[4], prefix = "df_train_test_")

In [ ]:
# df_train_test.head(10)

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Nettoyage, optimisation et sauvegarde du fichier final</span>

On va nettoyer un dernier coup puis on pourra resplitter en train et test et supprimer la colonne subset et la colonne target dans test.

<span style="color:blue;font-weight:bold"> Nettoyage, optimisation</span>

In [ ]:
# on remplace les inf par nan
df_train_test = clean_infinites(df_train_test)
# On nettoye les colonnes vides
df_train_test,dropped_cols = drop_empty_columns(df_train_test,0.8)
print(f'colonnes supprimées:\n{dropped_cols}')
# Les colonnes a valeur unique
df_train_test, dropped_unique_value = drop_col_with_unique_value(df_train_test)


# On optimise le format et on transforme cette fois-ci les object en category
dr_train_test = optimize_dtypes(df_train_test,True)
# on trnsforme les faux float en int
df_train_test = float_to_int(df_train_test)

<span style="color:blue;font-weight:bold"> Split train/test</span>

In [ ]:
# On sauvegarde temporairement train_test
export_datas(df_train_test, datas_path, step = 'final_datasets', prefix='temp_train_test')

# on créer le fichier train
df_train_test = df_train_test[df_train_test['subset'] == 'train']
df_train_test.drop(columns=['subset'], inplace=True)
# On sauvegarde le résultat final pour le Train
export_datas(df_train_test, datas_path, step = 'final_datasets', prefix='train')
# On supprime le dataset en mémoire
del df_train_test
gc.collect()

# On recharge train_test et on crée le fichier test
temp_file_path_final_datasets = \
    datas_path/'final_datasets'/f"temp_train_test_final_datasets.parquet"
df_train_test = pd.read_parquet(temp_file_path_final_datasets)
df_train_test = df_train_test[df_train_test['subset'] == 'test']
df_train_test.drop(columns=['subset', 'TARGET'], inplace=True)
# On sauvegarde le résultat final pour le Test
export_datas(df_train_test, datas_path, step = 'final_datasets', prefix='test')
# On supprime le dataset en mémoire
del df_train_test
gc.collect()



On jette un dernier regard sur les fichiers obtenus (on peut décommenter les df_train_test.head(10) avant de compiler pour voir l'évolution de la dataframe).
**On n'ouvrira que train.parquet**

In [ ]:
df_train = pd.read_parquet(datas_path/'final_datasets/train.parquet')

In [ ]:
df_train.info()

In [ ]:
df_train.head(10)